# Elo Rating Engine

Tujuan:

Notebook ini bertujuan membangun **Elo Rating Engine** yang digunakan untuk menghitung kekuatan relatif setiap tim nasional berdasarkan riwayat pertandingan internasional.
Berbeda dengan *historical feature engineering* yang menghasilkan statistik sederhana seperti *win rate* atau rata-rata gol, Elo Rating merupakan sistem pemeringkatan dinamis yang memperbarui kekuatan setiap tim setelah setiap pertandingan selesai.

Seluruh proses perhitungan dilakukan secara **kronologis**, sehingga rating yang digunakan pada suatu pertandingan hanya berasal dari informasi yang tersedia sebelum pertandingan tersebut dimainkan. Pendekatan ini memastikan bahwa proses pembangunan fitur tetap bebas dari *future data leakage* dan merepresentasikan kondisi prediksi di dunia nyata (*real-world prediction*).
Output utama dari notebook ini adalah histori Elo Rating setiap tim yang selanjutnya akan digunakan pada tahap **Elo Feature Engineering** untuk membangun fitur-fitur prediktif seperti `home_elo`, `away_elo`, dan `elo_difference`.

## 1. Loading

Pertama, kita akan memuat dataset hasil *data preprocessing* yang akan digunakan sebagai dasar perhitungan Elo Rating.
Pada notebook ini hanya beberapa kolom utama yang diperlukan, yaitu informasi tanggal pertandingan, nama kedua tim, serta skor akhir pertandingan.

In [1]:
import pandas as pd
from collections import defaultdict

In [2]:
results = pd.read_csv("../data/interim/results_clean.csv")

In [ ]:
# AMBIL YANG DIBUTUHKAN
results = results[
    [
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
]

In [ ]:
# KONVERSI DATE
results["date"] = pd.to_datetime(results["date"])

In [ ]:
# SORT BY DATE
results = results.sort_values("date").reset_index(drop=True)

In [ ]:
# LIHAT
results.head()

,date,home_team,away_team,home_score,away_score
0,1872-11-30,Scotland,England,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0
2,1874-03-07,Scotland,England,2.0,1.0
3,1875-03-06,England,Scotland,2.0,2.0
4,1876-03-04,Scotland,England,3.0,0.0


Dataset berhasil dimuat dan hanya menyisakan kolom-kolom yang diperlukan untuk membangun Elo Rating. Seluruh pertandingan kemudian diurutkan berdasarkan tanggal guna memastikan proses pembaruan rating mengikuti urutan kronologis. Langkah ini sangat penting karena Elo Rating merupakan sistem yang bergantung pada hasil pertandingan sebelumnya. Dengan demikian, setiap pembaruan rating hanya memanfaatkan informasi yang memang telah tersedia pada saat pertandingan berlangsung.

## 2. Elo Rating Concept

Sebelum membangun Elo Rating Engine, penting untuk memahami konsep dasar dari sistem Elo Rating itu sendiri. Elo Rating merupakan sistem pemeringkatan yang awalnya dikembangkan oleh **Arpad Elo** untuk mengukur kekuatan relatif pemain catur. Seiring waktu, metode ini diadaptasi ke berbagai cabang olahraga, termasuk sepak bola, karena mampu menggambarkan perubahan kekuatan tim secara dinamis berdasarkan hasil pertandingan. Berbeda dengan statistik sederhana seperti *win rate* atau rata-rata gol, Elo Rating tidak hanya mempertimbangkan hasil pertandingan, tetapi juga memperhitungkan kekuatan lawan yang dihadapi. Sebagai contoh, kemenangan atas tim yang memiliki rating tinggi akan memberikan peningkatan rating yang lebih besar dibandingkan kemenangan atas tim dengan rating yang lebih rendah. Sebaliknya, jika sebuah tim dengan rating tinggi kalah dari tim yang jauh lebih lemah, maka penurunan rating yang diterima juga akan lebih besar. Dengan mekanisme tersebut, Elo Rating mampu memberikan representasi kekuatan tim yang terus berkembang mengikuti performa aktual sepanjang waktu.

### 2.1 Elo Rating Workflow
Secara umum, proses pembaruan Elo Rating pada setiap pertandingan mengikuti alur berikut:
1. Mengambil rating kedua tim sebelum pertandingan dimulai.
2. Menghitung probabilitas kemenangan (*expected score*) berdasarkan selisih rating kedua tim.
3. Mengamati hasil pertandingan sebenarnya (*actual score*).
4. Memperbarui rating kedua tim menggunakan rumus Elo.
5. Menyimpan rating terbaru untuk digunakan pada pertandingan berikutnya.

Karena setiap pembaruan selalu menggunakan rating sebelum pertandingan berlangsung, sistem ini secara alami mengikuti urutan kronologis dan tidak memanfaatkan informasi dari masa depan (*future data leakage*).

### 2.2 Elo Rating Formula

Proses pembaruan Elo Rating terdiri dari tiga komponen utama:

- **Expected Score**, yaitu probabilitas kemenangan berdasarkan selisih rating kedua tim.
- **Actual Score**, yaitu hasil pertandingan sebenarnya.
- **Rating Update**, yaitu perhitungan rating baru.

Persamaan Elo dituliskan sebagai berikut:

$$
R_{new} = R_{old} + K \times (S - E)
$$

dengan:

- $R_{old}$ = rating sebelum pertandingan
- $R_{new}$ = rating setelah pertandingan
- $K$ = K-Factor
- $S$ = Actual Score
- $E$ = Expected Score

Nilai **Actual Score** ditentukan sebagai berikut:

| Hasil Pertandingan | Nilai |
|--------------------|------:|
| Menang | 1.0 |
| Seri | 0.5 |
| Kalah | 0.0 |

Semakin besar perbedaan antara hasil yang diperkirakan (*Expected Score*) dan hasil yang sebenarnya (*Actual Score*), semakin besar pula perubahan Elo Rating yang terjadi.